# CEG P2 真实图像生成 Colab Notebook

该 Notebook 用于在 Colab GPU 环境中执行 `p2_image_generation_outputs` 阶段。

重要边界:
- Notebook 只做环境准备和脚本编排, 不直接手写正式 `prompt_plan.json`、`image_pairs.json` 或 image manifests。
- 真正的图像生成由你配置的外部 SD / watermark backend 完成。
- P2 是否完成只以 `validate_pilot_image_generation_outputs.py --require-pass` 的结果为准。
- 不要把 Hugging Face token 写入 Notebook、JSON、CSV、manifest 或日志。token 应只存在于 Colab 环境变量或 Colab secret 中。


In [ ]:
# 1. 配置区
from pathlib import Path
import json
import os
import subprocess
import sys

# 如果需要从远端 clone 仓库, 填写仓库 URL; 如果已经把仓库上传或挂载到 /content/CEG, 可留空。
REPO_URL = ""
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")

DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
PILOT_WORKSPACE_ROOT = DRIVE_ROOT / "pilot_runs" / "real_pilot_input_workspace_20260617_034500"

COMMAND_FILE = PILOT_WORKSPACE_ROOT / "configs" / "p2_external_backend_command.draft.json"
IMAGE_OUTPUT_ROOT = PILOT_WORKSPACE_ROOT / "inputs" / "images"
PROMPT_PLAN = PILOT_WORKSPACE_ROOT / "inputs" / "prompts" / "prompt_plan.draft.json"
MODEL_CONFIG = PILOT_WORKSPACE_ROOT / "configs" / "model_config.draft.json"

# 若你已经手工把 COMMAND_FILE 中 external_command_placeholder 替换为 external_command, 保持 False。
# 若希望本 Notebook 写入 external_command, 设置为 True 并填写 EXTERNAL_COMMAND。
APPLY_EXTERNAL_COMMAND_FROM_NOTEBOOK = False
EXTERNAL_COMMAND = [
    "python",
    "/content/replace_with_real_sd_watermark_backend.py",
    "--prompt-plan",
    str(PROMPT_PLAN),
    "--out",
    str(IMAGE_OUTPUT_ROOT),
    "--model-config",
    str(MODEL_CONFIG),
]

# 运行 P2 包装入口。只有命令文件校验通过后才应设置为 True。
RUN_P2_IMAGE_GENERATION = False

# 默认与 RUN_P2_IMAGE_GENERATION 保持一致。False 时允许整本 Notebook 做只读 dry-run。
REQUIRE_BACKEND_COMMAND_READY = RUN_P2_IMAGE_GENERATION

# P2 成功后可生成 P3/P4 接续计划。该步骤不运行 attack 或 detection, 只写 runbook。
BUILD_POST_P2_RESUME_PLAN = True


In [ ]:
# 2. 挂载 Google Drive 并准备仓库
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if REPO_URL and not REPO_ROOT.exists():
    clone_cmd = ["git", "clone"]
    if REPO_BRANCH:
        clone_cmd += ["--branch", REPO_BRANCH]
    clone_cmd += [REPO_URL, str(REPO_ROOT)]
    subprocess.run(clone_cmd, check=True)

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"未找到仓库目录: {REPO_ROOT}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo_root =", REPO_ROOT)
print("workspace =", PILOT_WORKSPACE_ROOT)


In [ ]:
# 3. 检查 P2 输入和命令文件
required_paths = [
    PROMPT_PLAN,
    MODEL_CONFIG,
    COMMAND_FILE,
    PILOT_WORKSPACE_ROOT / "pilot_p0_input_freeze_report.json",
    PILOT_WORKSPACE_ROOT / "pilot_image_generation_launch_plan_report.json",
]
for path in required_paths:
    print(("OK" if path.exists() else "MISSING"), path)

if COMMAND_FILE.exists():
    payload = json.loads(COMMAND_FILE.read_text(encoding="utf-8-sig"))
    print("command_file_status =", payload.get("manifest_status"))
    print("has_external_command =", "external_command" in payload)
    print("has_placeholder =", "external_command_placeholder" in payload)


In [ ]:
# 4. 可选: 将真实外部 backend 命令写入命令文件
# 注意: 不要把 HF_TOKEN 或其他密钥写入 EXTERNAL_COMMAND。
if APPLY_EXTERNAL_COMMAND_FROM_NOTEBOOK:
    apply_cmd = [
        sys.executable,
        "scripts/apply_pilot_image_generation_backend_command.py",
        "--command-file",
        str(COMMAND_FILE),
        "--external-command-json",
        json.dumps(EXTERNAL_COMMAND, ensure_ascii=False),
        "--require-ready",
    ]
    print("运行:", " ".join(apply_cmd))
    subprocess.run(apply_cmd, check=True)
else:
    print("跳过命令文件写入。请确认 COMMAND_FILE 已由你手工替换为 external_command。")


In [ ]:
# 5. 校验外部 backend 命令文件
validate_command = [
    sys.executable,
    "scripts/validate_pilot_image_generation_backend_command.py",
    "--command-file",
    str(COMMAND_FILE),
]
if REQUIRE_BACKEND_COMMAND_READY:
    validate_command.append("--require-ready")
print("运行:", " ".join(validate_command))
subprocess.run(validate_command, check=True)


In [ ]:
# 6. 执行 P2 图像生成包装入口
# 该步骤会调用 COMMAND_FILE 中的真实外部 SD / watermark backend。
# 如果 RUN_P2_IMAGE_GENERATION 为 False, 本 cell 只打印应执行的命令。
p2_cmd = [
    sys.executable,
    "scripts/run_pilot_image_generation_backend.py",
    "--prompt-plan",
    str(PROMPT_PLAN),
    "--out",
    str(IMAGE_OUTPUT_ROOT),
    "--model-config",
    str(MODEL_CONFIG),
    "--external-command-json-file",
    str(COMMAND_FILE),
    "--require-pass",
]
print("P2 命令:")
print(" ".join(p2_cmd))
if RUN_P2_IMAGE_GENERATION:
    subprocess.run(p2_cmd, check=True)
else:
    print("RUN_P2_IMAGE_GENERATION = False, 已跳过真实 GPU 执行。")


In [ ]:
# 7. 验收 P2 输出
# 只有该命令通过, 才能认为 p2_image_generation_outputs 已完成。
p2_acceptance = [
    sys.executable,
    "scripts/validate_pilot_image_generation_outputs.py",
    "--output-root",
    str(IMAGE_OUTPUT_ROOT),
    "--out",
    str(PILOT_WORKSPACE_ROOT / "pilot_image_generation_output_acceptance_report.json"),
    "--require-pass",
]
print("运行:", " ".join(p2_acceptance))
if RUN_P2_IMAGE_GENERATION:
    subprocess.run(p2_acceptance, check=True)
else:
    print("RUN_P2_IMAGE_GENERATION = False, 已跳过 P2 通过性验收。")


In [ ]:
# 8. 刷新阶段进度, 并生成 P2 后接续计划
summary_cmd = [
    sys.executable,
    "scripts/build_pilot_stage_progress_summary.py",
    "--workspace",
    str(PILOT_WORKSPACE_ROOT),
]
print("运行:", " ".join(summary_cmd))
subprocess.run(summary_cmd, check=True)

if BUILD_POST_P2_RESUME_PLAN:
    resume_cmd = [
        sys.executable,
        "scripts/build_pilot_post_p2_resume_plan.py",
        "--workspace",
        str(PILOT_WORKSPACE_ROOT),
    ]
    print("运行:", " ".join(resume_cmd))
    subprocess.run(resume_cmd, check=True)


In [ ]:
# 9. 显示关键结果路径
paths = {
    "p2_acceptance_report": PILOT_WORKSPACE_ROOT / "pilot_image_generation_output_acceptance_report.json",
    "stage_summary": PILOT_WORKSPACE_ROOT / "pilot_stage_progress_summary.json",
    "post_p2_resume_runbook": PILOT_WORKSPACE_ROOT / "gpu_handoff" / "post_p2_resume" / "pilot_post_p2_resume_runbook.md",
    "image_pairs": IMAGE_OUTPUT_ROOT / "image_pairs.json",
    "clean_dir": IMAGE_OUTPUT_ROOT / "clean",
    "watermarked_dir": IMAGE_OUTPUT_ROOT / "watermarked",
}
for name, path in paths.items():
    print(name, "=", path, "exists=", path.exists())
